# Task 4 – Netflix Titles Dataset: Exploratory Analysis & Insights

Explore Netflix metadata, clean date and multi-label fields, engineer features, visualize content patterns, and interpret findings.

**Run instructions:** Keep the original CSV in the same folder as this notebook, then run cells from top to bottom.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
df=pd.read_csv("netflix_titles.csv")
df.head()

## 1. Data Understanding & Cleaning

In [ ]:
df.info()
df.describe(include="all").T
df.isna().sum()
df["date_added"]=pd.to_datetime(df["date_added"].astype(str).str.strip(),errors="coerce")
for c in ["director","cast","country"]:
    df[c]=df[c].fillna("Unknown")
df["rating"]=df["rating"].fillna("Not Rated")
df["release_year"]=pd.to_numeric(df["release_year"],errors="coerce").astype("Int64")

## 2. Feature Engineering

In [ ]:
df["release_decade"]=(df["release_year"]//10*10).astype("Int64")
df["is_movie"]=(df["type"]=="Movie").astype(int)
df["content_type"]=df["type"]
df["date_added_year"]=df["date_added"].dt.year.astype("Int64")
df[["title","type","release_year","release_decade","is_movie","date_added","date_added_year"]].head()

## 3. Multi-label Analysis

In [ ]:
genre_counts=df["listed_in"].dropna().str.split(", ").explode().value_counts()
country_counts=df.loc[df["country"]!="Unknown","country"].str.split(", ").explode().value_counts()
top_countries=country_counts.head(5)
type_counts=df["type"].value_counts()
year_counts=df.groupby("release_year",dropna=True).size().sort_index()
print("Top genres:\n",genre_counts.head(10))
print("Top countries:\n",top_countries)

## 4. Visualizations

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(14,10))
type_counts.plot.pie(autopct="%1.1f%%",startangle=90,ax=ax[0,0],ylabel="",title="Movies vs TV Shows")
year_counts.tail(35).plot(ax=ax[0,1],marker="o"); ax[0,1].set_title("Titles by Release Year"); ax[0,1].set_xlabel("Release year")
genre_counts.head(10).sort_values().plot.barh(ax=ax[1,0],color="slateblue"); ax[1,0].set_title("Top 10 Genres")
rows=[]
for _,r in df.iterrows():
    if r["country"]!="Unknown": rows.extend((c,r["type"]) for c in str(r["country"]).split(", "))
ct=pd.DataFrame(rows,columns=["Country","Type"]); pivot=ct[ct["Country"].isin(top_countries.index)].pivot_table(index="Country",columns="Type",aggfunc="size",fill_value=0)
sns.heatmap(pivot,annot=True,fmt="d",cmap="Blues",ax=ax[1,1]); ax[1,1].set_title("Top 5 Countries: Content Type Volume")
plt.tight_layout(); plt.show()

## 5. Insights & Exploratory Trend Baseline

In [ ]:
print("Movies vs TV Shows:\n",type_counts)
print("Top 10 genres:\n",genre_counts.head(10))
print("Titles by release year (latest):\n",year_counts.tail(10))
recent=year_counts[year_counts.index>=year_counts.index.max()-9].dropna()
if len(recent)>=3:
    slope,intercept=np.polyfit(recent.index.astype(int),recent.to_numpy(),1)
    future=[int(recent.index.max()+1),int(recent.index.max()+2)]
    print("Simple linear extrapolation (not a reliable forecast):", {y:max(0,int(round(slope*y+intercept))) for y in future})

In [ ]:
df.to_csv("netflix_titles_cleaned.csv",index=False)
print("Saved cleaned Netflix CSV.")